# 📚 UPPA Smart Assistant — Notebook 02 (version 1.1)
## Chunking robuste, embeddings et indexation Qdrant

**Mini-projet M2 — Industry 4.0 & Logistique / services**

Cette version améliore le découpage des **grands tableaux**, évite d’indexer les pages de navigation du catalogue, plafonne les visualisations et introduit la **diversification des sources et des domaines**. Elle prépare un jeu de questions annotées pour mesurer la pertinence sur les documents réellement collectés.

L'archive exportée reste compatible avec le Notebook 03.

### Compétences visées

1. Importer et **valider le schéma JSONL** produit par l'atelier précédent.
2. Distinguer **document, section, chunk, embedding et point Qdrant**.
3. Comparer le découpage récursif générique à un découpage **respectant les titres et tableaux**.
4. Calculer des embeddings français/multilingues ; interpréter le **cosinus**.
5. Configurer Qdrant, indexer des vecteurs et leurs **payloads** (métadonnées).
6. Rechercher avec et sans **filtrage par domaine**, citer les sources récupérées.
7. Vérifier et exporter une base indexée **reproductible**, prête pour le Notebook 03.

**Pré-requis :** Notebook 01 terminé ou corpus de secours fictif inclus ; Python et notions d'embeddings vues aux cours 1–2.

**Pratique Colab :** exécution CPU possible, sans Docker et sans clé payante. Internet est nécessaire à l'installation des bibliothèques et au premier téléchargement du modèle d'embeddings.

In [ ]:
from IPython.display import display, HTML

display(HTML("""
<div style="font-family:system-ui;max-width:1060px;margin:8px auto 20px;background:#f8f6ee;border:1px solid #d8d9cc;border-radius:22px;padding:24px;color:#203a32">
 <div style="font-size:12px;text-transform:uppercase;letter-spacing:.15em;color:#ae6346;font-weight:700">Schéma pédagogique · construction de l'index</div>
 <div style="display:flex;flex-wrap:wrap;gap:8px;align-items:center;margin-top:16px;font-size:13px">
  <div style="background:#eadac4;padding:13px;border-radius:13px;flex:1;min-width:135px"><b>01 · Corpus</b><br>JSONL issu du Notebook 01</div>
  <b>→</b><div style="background:#dfe9e4;padding:13px;border-radius:13px;flex:1;min-width:135px"><b>02 · Sections</b><br>Titres, listes, tableaux</div>
  <b>→</b><div style="background:#fff;border:1px solid #d8d9cc;padding:13px;border-radius:13px;flex:1;min-width:135px"><b>03 · Chunks</b><br>Texte + métadonnées</div>
  <b>→</b><div style="background:#dfe9e4;padding:13px;border-radius:13px;flex:1;min-width:135px"><b>04 · Embeddings</b><br>Vecteurs normalisés</div>
  <b>→</b><div style="background:#eadac4;padding:13px;border-radius:13px;flex:1;min-width:135px"><b>05 · Qdrant</b><br>Index + payloads</div>
 </div>
 <div style="margin-top:12px;color:#63756c;font-size:13px"><b>Au moment de la question :</b> Question → vecteur → similarité cosinus → top-k → extraits + URL (le LLM viendra dans le Notebook 03).</div>
</div>"""))

---
## Étape 0 — Installation et environnement

Nous utilisons **`langchain-text-splitters`** pour le découpage, **`sentence-transformers`** pour la représentation multilingue et **`qdrant-client`** pour la base vectorielle. Qdrant possède un mode **local persistant** (`QdrantClient(path=...)`) : il ne nécessite pas de serveur lors des TP.

📌 Si Colab demande de redémarrer le runtime après une installation, redémarrez-le puis relancez les cellules dans l'ordre.

In [ ]:
%pip -q install "langchain-text-splitters>=0.3,<2" "sentence-transformers>=3,<6" "qdrant-client>=1.12,<2" "pandas>=2" "matplotlib>=3.7"

In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
from urllib.parse import urlparse
from IPython.display import display, Markdown
import json, hashlib, io, re, uuid, zipfile, shutil, sys, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models

BASE = Path('/content/UPPA_RAG_NB02' if Path('/content').exists() else './UPPA_RAG_NB02').resolve()
EXPORT_DIR = BASE/'exports'
DB_DIR = BASE/'qdrant_local'
for d in (BASE, EXPORT_DIR): d.mkdir(parents=True, exist_ok=True)
print('Dossier de travail :', BASE)
print('Python :',platform.python_version())

---
## Étape 1 — Importer le corpus du Notebook 01

**Deux situations :**

- **Travail réel** : utiliser l'archive ZIP exportée par le Notebook 01, par exemple `UPPA_Notebook01_exports_uppa.zip`, contenant `exports/corpus_uppa.jsonl` et `exports/dataset_info.json`.
- **Découverte pédagogique** : si aucun fichier n'est fourni, utiliser les **six documents fictifs intégrés**, provenant de la structure d'export du Notebook 01.

**Dans Colab**, vous pouvez charger votre ZIP avec la cellule facultative ci-dessous, puis renseigner le nom du fichier dans `INPUT_FILE`. Un *runtime Colab* ne conserve pas automatiquement les fichiers de la session précédente : exportez ou sauvegardez l'archive du Notebook 01.

In [ ]:
# Réglages utilisateur (rien à modifier pour une première démonstration).
INPUT_FILE = ""      # Exemples : "/content/UPPA_Notebook01_exports_uppa.zip" ou "/content/corpus_uppa.jsonl"
UPLOAD_WITH_COLAB = False  # Mettre True pour ouvrir le sélecteur de fichiers Colab.

if UPLOAD_WITH_COLAB:
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError('Cette option de chargement est réservée à Google Colab.') from exc
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Sélectionnez une seule archive ZIP ou un seul corpus JSONL.')
    INPUT_FILE = str(Path.cwd() / next(iter(uploaded.keys())))
    print('Fichier reçu :', INPUT_FILE)

In [ ]:
DEMO_DOCS_NB01 = json.loads(r'''[
  {
    "document_id": "demo_ffe9847450c745d3",
    "domain": "formation",
    "document_type": "fiction",
    "title": "Master informatique — exemple fictif",
    "source_url": null,
    "collected_at": "2026-11-09T23:05:47+00:00",
    "dataset_kind": "synthetic_demo",
    "language": "fr",
    "academic_years_detected": [],
    "headings": [
      {
        "level": 1,
        "text": "Master informatique — exemple fictif"
      },
      {
        "level": 2,
        "text": "Présentation"
      },
      {
        "level": 2,
        "text": "Parcours"
      },
      {
        "level": 2,
        "text": "Compétences"
      }
    ],
    "links": [],
    "table_count": 0,
    "content_markdown": "# Master informatique — exemple fictif\n\n## Présentation\n\nFormation expérimentale qui illustre les systèmes d'information, l'IA et l'informatique industrielle.\n\n## Parcours\n\n* Industry 4.0 — cas fictif\n* Logistique et services — cas fictif\n\n## Compétences\n\nConcevoir des applications et analyser des données.",
    "content_text": "Master informatique — exemple fictif Présentation Formation expérimentale qui illustre les systèmes d'information, l'IA et l'informatique industrielle. Parcours Industry 4.0 — cas fictif Logistique et services — cas fictif Compétences Concevoir des applications et analyser des données.",
    "sha256_text": "23fe2e187b2e3925a1ebf92371bcc7c2d08bf38b09896ab9e28d8c6e7769df3c",
    "character_count": 286,
    "word_count": 37
  },
  {
    "document_id": "demo_494779650bc67814",
    "domain": "formation",
    "document_type": "fiction",
    "title": "Parcours Industry 4.0 — exemple fictif",
    "source_url": null,
    "collected_at": "2026-11-09T23:05:47+00:00",
    "dataset_kind": "synthetic_demo",
    "language": "fr",
    "academic_years_detected": [],
    "headings": [
      {
        "level": 1,
        "text": "Parcours Industry 4.0 — exemple fictif"
      },
      {
        "level": 2,
        "text": "Objectifs"
      },
      {
        "level": 2,
        "text": "Tableau des unités"
      }
    ],
    "links": [],
    "table_count": 1,
    "content_markdown": "# Parcours Industry 4.0 — exemple fictif\n\n## Objectifs\n\nExplorer la maintenance prédictive, les capteurs et le pilotage de production.\n\n## Tableau des unités\n\n| UE | Thème |\n| --- | --- |\n| UE1 | Capteurs |\n| UE2 | Données industrielles |",
    "content_text": "Parcours Industry 4.0 — exemple fictif Objectifs Explorer la maintenance prédictive, les capteurs et le pilotage de production. Tableau des unités UE Thème UE1 Capteurs UE2 Données industrielles",
    "sha256_text": "374260d1e7cf7469f0b9362b90e1098f570027f0e31ef7019c2cd26a708b15e4",
    "character_count": 194,
    "word_count": 28
  },
  {
    "document_id": "demo_c593913670f65f7d",
    "domain": "admission",
    "document_type": "fiction",
    "title": "Admission en master — exemple fictif",
    "source_url": null,
    "collected_at": "2026-11-09T23:05:47+00:00",
    "dataset_kind": "synthetic_demo",
    "language": "fr",
    "academic_years_detected": [],
    "headings": [
      {
        "level": 1,
        "text": "Admission en master — exemple fictif"
      },
      {
        "level": 2,
        "text": "Documents"
      },
      {
        "level": 2,
        "text": "Procédure"
      }
    ],
    "links": [],
    "table_count": 0,
    "content_markdown": "# Admission en master — exemple fictif\n\n## Documents\n\n* Curriculum vitae\n* Relevés de notes\n\n## Procédure\n\nConsulter les modalités officielles, différentes selon le profil et l'année.\n\nLes dates citées dans ce document sont fictives.",
    "content_text": "Admission en master — exemple fictif Documents Curriculum vitae Relevés de notes Procédure Consulter les modalités officielles, différentes selon le profil et l'année. Les dates citées dans ce document sont fictives.",
    "sha256_text": "dc105ec8fbb47a89d7e54543798866444e2a09f885ad9f2050c74d868c263c96",
    "character_count": 216,
    "word_count": 31
  },
  {
    "document_id": "demo_d1e2dbcf93b1520d",
    "domain": "admission",
    "document_type": "fiction",
    "title": "Calendrier d'admission — exemple fictif",
    "source_url": null,
    "collected_at": "2026-11-09T23:05:47+00:00",
    "dataset_kind": "synthetic_demo",
    "language": "fr",
    "academic_years_detected": [],
    "headings": [
      {
        "level": 1,
        "text": "Calendrier d'admission — exemple fictif"
      },
      {
        "level": 2,
        "text": "Dates illustratives"
      }
    ],
    "links": [],
    "table_count": 1,
    "content_markdown": "# Calendrier d'admission — exemple fictif\n\n## Dates illustratives\n\n| Phase | Date fictive |\n| --- | --- |\n| Candidature | 15 janvier 2030 |\n| Réponse | 20 mars 2030 |\n\nCes dates ne concernent aucune campagne officielle.",
    "content_text": "Calendrier d'admission — exemple fictif Dates illustratives Phase Date fictive Candidature 15 janvier 2030 Réponse 20 mars 2030 Ces dates ne concernent aucune campagne officielle.",
    "sha256_text": "6e16b05ee700b7b029fefc5fa48ee8283e18dc69b07f6f5d7128b2f265ab1f09",
    "character_count": 179,
    "word_count": 25
  },
  {
    "document_id": "demo_432ebb2fd9f03e32",
    "domain": "recherche",
    "document_type": "fiction",
    "title": "Laboratoire Systèmes numériques — exemple fictif",
    "source_url": null,
    "collected_at": "2026-11-09T23:05:47+00:00",
    "dataset_kind": "synthetic_demo",
    "language": "fr",
    "academic_years_detected": [],
    "headings": [
      {
        "level": 1,
        "text": "Laboratoire Systèmes numériques — exemple fictif"
      },
      {
        "level": 2,
        "text": "Axes de recherche"
      },
      {
        "level": 2,
        "text": "Équipes"
      }
    ],
    "links": [],
    "table_count": 0,
    "content_markdown": "# Laboratoire Systèmes numériques — exemple fictif\n\n## Axes de recherche\n\nRéseaux, intelligence artificielle, simulation et systèmes cyber-physiques.\n\n## Équipes\n\nEquipe Méthodes et Données.",
    "content_text": "Laboratoire Systèmes numériques — exemple fictif Axes de recherche Réseaux, intelligence artificielle, simulation et systèmes cyber-physiques. Équipes Equipe Méthodes et Données.",
    "sha256_text": "418165d42fc1434b2e1deed1aaa7bbb3422d4587787cd75a442734c366c8857a",
    "character_count": 178,
    "word_count": 21
  },
  {
    "document_id": "demo_f52ce7bf39c1439e",
    "domain": "recherche",
    "document_type": "fiction",
    "title": "Projet Logistique intelligente — exemple fictif",
    "source_url": null,
    "collected_at": "2026-11-09T23:05:47+00:00",
    "dataset_kind": "synthetic_demo",
    "language": "fr",
    "academic_years_detected": [],
    "headings": [
      {
        "level": 1,
        "text": "Projet Logistique intelligente — exemple fictif"
      },
      {
        "level": 2,
        "text": "Thématiques"
      },
      {
        "level": 2,
        "text": "Partenariats"
      }
    ],
    "links": [],
    "table_count": 0,
    "content_markdown": "# Projet Logistique intelligente — exemple fictif\n\n## Thématiques\n\n* Traçabilité\n* Optimisation d'entrepôt\n* Recommandation de services\n\n## Partenariats\n\nExemple pédagogique sans partenaires réels.",
    "content_text": "Projet Logistique intelligente — exemple fictif Thématiques Traçabilité Optimisation d'entrepôt Recommandation de services Partenariats Exemple pédagogique sans partenaires réels.",
    "sha256_text": "6bd67d9f9953a9013bd8ef18836061e4d62c78f486bc356bb635baa81ca1e0a2",
    "character_count": 179,
    "word_count": 19
  }
]''')

print('Exemples du Notebook 01 inclus :',len(DEMO_DOCS_NB01), 'documents fictifs')
assert len(DEMO_DOCS_NB01) == 6
assert {d['domain'] for d in DEMO_DOCS_NB01} == {'formation','admission','recherche'}

In [ ]:
ALLOWED_DOMAINS = {'formation','admission','recherche'}
ALLOWED_HOSTS = {'formation.univ-pau.fr','recherche.univ-pau.fr'}
MAX_JSONL_BYTES = 15_000_000
MAX_DOCUMENTS = 1500


def read_jsonl_lines(data: bytes):
    if len(data) > MAX_JSONL_BYTES:
        raise ValueError('Fichier de corpus trop volumineux pour cet atelier.')
    return [json.loads(line) for line in data.decode('utf-8-sig').splitlines() if line.strip()]


def load_nb01_file(path):
    path=Path(path)
    if not path.exists(): raise FileNotFoundError(path)
    if path.suffix.lower()=='.zip':
        with zipfile.ZipFile(path) as z:
            entries=[name for name in z.namelist() if re.search(r'(?:^|/)corpus_(?:uppa|synthetic_demo)\.jsonl$',name)]
            if len(entries)!=1: raise ValueError(f'Une seule entrée corpus_*.jsonl attendue, obtenu {entries}')
            info=z.getinfo(entries[0])
            if info.file_size > MAX_JSONL_BYTES: raise ValueError('Fichier JSONL interne trop volumineux.')
            docs=read_jsonl_lines(z.read(entries[0]))
            return docs, f'{path.name}::{entries[0]}'
    if path.suffix.lower()=='.jsonl':
        return read_jsonl_lines(path.read_bytes()), str(path)
    raise ValueError('Format accepté : .zip exporté par Notebook 01 ou corpus_*.jsonl')


def validate_documents(docs):
    if not isinstance(docs,list) or not docs or len(docs)>MAX_DOCUMENTS:
        raise ValueError('Corpus vide ou dépassement du nombre de documents autorisé.')
    keys={'document_id','domain','title','content_markdown','dataset_kind','source_url'}
    kinds=set()
    ids=[]
    for d in docs:
        if not isinstance(d,dict) or not keys.issubset(d):
            raise ValueError('Schéma incompatible avec le Notebook 01.')
        if d['domain'] not in ALLOWED_DOMAINS: raise ValueError('Domaine non reconnu')
        if not isinstance(d['content_markdown'],str) or not d['content_markdown'].strip():
            raise ValueError('Un document est vide')
        if len(d['content_markdown']) > 600_000: raise ValueError('Document trop long')
        kinds.add(d['dataset_kind'])
        ids.append(d['document_id'])
        if d['dataset_kind']=='live':
            url=d.get('source_url') or ''
            parsed=urlparse(url)
            if parsed.scheme!='https' or parsed.hostname not in ALLOWED_HOSTS:
                raise ValueError('Document live sans URL source officielle vérifiable.')
    if len(ids)!=len(set(ids)): raise ValueError('document_id répété : réviser les doublons du Notebook 01.')
    if kinds not in ({'live'}, {'synthetic_demo'}):
        raise ValueError(f'Corpus mixte ou étiquette inconnue : {kinds}')
    return next(iter(kinds))

# Reprendre l'export de NB01 s'il est disponible dans la même session.
if not INPUT_FILE:
    search_roots=[Path('/content/UPPA_RAG'),Path('/content'),Path.cwd()/'UPPA_RAG']
    options=[]
    for root in search_roots:
        if root.exists():
            options += list(root.glob('UPPA_Notebook01_exports_uppa.zip'))
            options += list(root.glob('UPPA_Notebook01_exports_synthetic_demo.zip'))
    if options:
        options.sort(key=lambda p: ('synthetic_demo' in p.name, str(p)))
        INPUT_FILE=str(options[0])

if INPUT_FILE:
    DOCUMENTS, INPUT_ORIGIN=load_nb01_file(INPUT_FILE)
else:
    DOCUMENTS=list(DEMO_DOCS_NB01)
    INPUT_ORIGIN='démonstration fictive intégrée (d’après Notebook 01)'
DATASET_KIND=validate_documents(DOCUMENTS)
print('Source :',INPUT_ORIGIN)
print('Documents :',len(DOCUMENTS),'| domaines :',dict(Counter(d['domain'] for d in DOCUMENTS)))
print('Mode :',DATASET_KIND)
if DATASET_KIND!='live':
    print('⚠️ Mode fictif : aucun fait de ce corpus ne concerne les formations ou admissions réelles de l’UPPA.')

# Les pages de navigation (catalogue, portail) restent dans l'inventaire mais
# ne deviennent pas des chunks vectoriels. Compatibilité avec corpus NB01 anciens.
KNOWN_NAVIGATION_PATHS={
    '/fr/catalogue/toutes-les-formations.html', '/fr/trouver-votre-formation.html',
    '/fr/admission.html', '/fr/laboratoires.html'
}
def is_navigation_hub(doc):
    path=urlparse(doc.get('source_url') or '').path
    return (doc.get('document_role')=='navigation_hub' or path in KNOWN_NAVIGATION_PATHS)

INDEX_DOCUMENTS=[d for d in DOCUMENTS if not is_navigation_hub(d)]
if not INDEX_DOCUMENTS:
    raise ValueError('Aucun document indexable : vérifier les URL et le nettoyage du Notebook 01.')
print('Documents dans le corpus :',len(DOCUMENTS))
print('Pages de navigation écartées de l’index :',len(DOCUMENTS)-len(INDEX_DOCUMENTS))
print('Documents réellement indexables :',len(INDEX_DOCUMENTS))


In [ ]:
overview = pd.DataFrame([{
  'domaine': d['domain'],
  'titre': d['title'],
  'caractères': len(d['content_markdown']),
  'tableaux_html_nb01': d.get('table_count',0),
  'source': d.get('source_url') or '(fictif)',
} for d in DOCUMENTS])
display(overview)
ax=overview.groupby('domaine')['caractères'].sum().sort_index().plot(kind='bar',rot=0,figsize=(7,3.1),title='Volume textuel par domaine (caractères)')
ax.set_ylabel('Caractères')
plt.tight_layout(); plt.show()

### Question pédagogique

**Pourquoi un même index peut-il contenir les trois domaines ?** Parce que chaque passage portera `domain`, `document_id`, `section`, `title` et `source_url`. La requête pourra ainsi filtrer un domaine sans mélanger les conditions d'admission avec des thèmes de laboratoire.

> À examiner : les documents du Notebook 01 ont été nettoyés, mais des informations institutionnelles peuvent être **historiques ou périmées**. Les métadonnées signalent leur date de collecte ; elles ne garantissent pas leur actualité.

---
## Étape 2 — Définir le chunking

Un **document** correspond à une page ou une fiche ; une **section** correspond à un titre du Markdown ; un **chunk** est l'unité réellement recherchée dans l'index.

Nous allons comparer :

**A — Chunking récursif simple.** On coupe le Markdown à taille limitée, avec chevauchement : méthode rapide, mais elle peut séparer une date de la ligne d'un tableau à laquelle elle se rapporte.

**B — Chunking structurel + récursif.** On isole d'abord les sections (`#`, `##`, `###`) et les tableaux Markdown. Le découpage récursif ne s'applique qu'au texte courant ; un tableau court reste entier ; un tableau plus long que le seuil est scindé **par groupes de lignes, en répétant ses en-têtes**, et une ligne géante est réécrite sous la forme `colonne : valeur`. Aucun chunk ne dépasse `CHUNK_SIZE`.

**Paramètres de départ :** `chunk_size = 800` caractères et `chunk_overlap = 120` caractères. **Ce sont des valeurs initiales à tester**, pas des normes universelles.

In [ ]:
CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
assert 0 <= CHUNK_OVERLAP < CHUNK_SIZE
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=['\n\n','\n','. ',' ', ''],
    keep_separator=True,
)

def chunking_simple(doc):
    return splitter.split_text(doc['content_markdown'])

ex_doc = max(DOCUMENTS, key=lambda d:len(d['content_markdown']))
simple_chunks = chunking_simple(ex_doc)
print('Document :',ex_doc['title'])
print('Nombre de morceaux récursifs simples :',len(simple_chunks))
print('Tailles :',[len(c) for c in simple_chunks])
print('Début du premier morceau :\n',simple_chunks[0][:450])

In [ ]:
ROW_LABEL_RESERVE = 48  # place réservée à l'étiquette « Tableau n, ligne m — »
HEADING = re.compile(r'^(#{1,6})\s+(.+?)\s*$',re.M)
TABLE_LINE = re.compile(r'^\s*\|.*\|\s*$')
TABLE_SEPARATOR = re.compile(r'^\s*\|?\s*:?-{3,}:?\s*(?:\|\s*:?-{3,}:?\s*)+\|?\s*$')

row_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE-ROW_LABEL_RESERVE, chunk_overlap=min(CHUNK_OVERLAP,80), length_function=len,
    separators=['\n','. ',' ',''], keep_separator=True)

def markdown_sections(markdown):
    sections=[]; hierarchy=[]; current=[]; current_label='Introduction'
    def flush():
        body='\n'.join(current).strip()
        if body: sections.append({'section':current_label,'body':body})
    for line in markdown.splitlines():
        found=HEADING.match(line)
        if found:
            flush(); current=[]
            level=len(found.group(1)); title=found.group(2).strip()
            hierarchy[:]=[item for item in hierarchy if item[0]<level]
            hierarchy.append((level,title))
            current_label=' / '.join(label for _,label in hierarchy)
        else:
            current.append(line)
    flush()
    return sections or ([{'section':current_label,'body':markdown.strip()}] if markdown.strip() else [])

def markdown_blocks(body):
    result=[]; current=[]; current_is_table=None
    def flush():
        if current:
            result.append({'kind':'table' if current_is_table else 'prose','text':'\n'.join(current).strip()})
    for line in body.splitlines():
        if not line.strip():
            flush(); current=[]; current_is_table=None; continue
        is_table=bool(TABLE_LINE.match(line))
        if current and current_is_table!=is_table:
            flush(); current=[]
        current_is_table=is_table
        current.append(line)
    flush()
    return [b for b in result if b['text']]

def split_large_table(table_text, section, table_index):
    """Conserve en-têtes et lignes ; découpe aussi une ligne surdimensionnée.

    Les fragments de lignes géantes sont étiquetés avec les noms des colonnes.
    Aucun bloc ne doit dépasser CHUNK_SIZE, même si un tableau dépasse 28k chars.
    """
    rows=[line.strip() for line in table_text.splitlines() if line.strip()]
    if not rows: return []
    sep_index=next((i for i,line in enumerate(rows[:3]) if TABLE_SEPARATOR.match(line)),None)
    if sep_index==1:
        header=rows[0]; separator=rows[1]; data=rows[2:]
    else:
        header=''; separator=''; data=rows
    prefix=(header+'\n'+separator+'\n') if header else ''
    out=[]; group=[]; first_idx=None
    def emit_group():
        nonlocal first_idx
        if group:
            text=prefix+'\n'.join(group)
            out.append({'section':section,'text':text,'kind':'table',
                        'table_index':table_index,'table_row_start':first_idx,'table_row_fragment':False})
            group.clear(); first_idx=None
    for ix,row in enumerate(data,1):
        draft=prefix+'\n'.join(group+[row])
        if len(draft)<=CHUNK_SIZE:
            if first_idx is None: first_idx=ix
            group.append(row)
            continue
        emit_group()
        if len(prefix+row)<=CHUNK_SIZE:
            group.append(row); first_idx=ix; continue
        # Un seul enregistrement dépasse la capacité (cellule géante ou
        # colonnes très longues) : exposer son contenu sous forme de fragments
        # portant l'indice de ligne et les noms des colonnes.
        headers=[x.strip() for x in header.strip('|').split('|')]
        values=[x.strip() for x in row.strip('|').split('|')]
        if headers and values:
            content=' ; '.join(f'{headers[i] if i<len(headers) else "Colonne "+str(i+1)} : {v}'
                              for i,v in enumerate(values))
        else:
            content=row
        label=f'Tableau {table_index}, ligne {ix} — '[:ROW_LABEL_RESERVE]
        # Splitter DÉDIÉ : sa taille tient déjà compte de l'étiquette. L'ancienne version
        # découpait à CHUNK_SIZE puis recoupait chaque morceau, ce qui créait des
        # mini-fragments orphelins (800 / 36 / 800 / 39 caractères) coupés en plein mot.
        for part in row_splitter.split_text(content):
            part=part.strip()
            if part:
                out.append({'section':section,'text':label+part,'kind':'table_row_fragment',
                            'table_index':table_index,'table_row_start':ix,'table_row_fragment':True})
    emit_group()
    # Cas limite : tableau constitué uniquement d'un en-tête trop long.
    if not out:
        for part in splitter.split_text(table_text):
            out.append({'section':section,'text':part[:CHUNK_SIZE],
                        'kind':'table_row_fragment','table_index':table_index,
                        'table_row_start':0,'table_row_fragment':True})
    return out

def chunk_structure(doc):
    chunks=[]; table_index=0
    for sec in markdown_sections(doc['content_markdown']):
        buffer=[]
        def flush_buffer():
            if not buffer: return
            for part in splitter.split_text('\n\n'.join(buffer)):
                if part.strip():
                    chunks.append({'section':sec['section'],'text':part.strip(),'kind':'prose'})
            buffer.clear()
        for block in markdown_blocks(sec['body']):
            if block['kind']=='table':
                flush_buffer(); table_index+=1
                chunks.extend(split_large_table(block['text'],sec['section'],table_index))
            else:
                buffer.append(block['text'])
        flush_buffer()
    return chunks

# Tests unitaires : tableaux courts ET tableau de 28 000 caractères.
short_table='| UE | Thème |\n| --- | --- |\n| UE1 | Capteurs |\n| UE2 | Data |'
short_parts=split_large_table(short_table,'Unités',1)
assert any('| UE1 | Capteurs |' in x['text'] for x in short_parts)
assert all(len(x['text'])<=CHUNK_SIZE for x in short_parts)
long_table='| Code | Description |\n| --- | --- |\n'+ '\n'.join(
    f'| M-{i:03d} | '+('Données scientifiques et ingénierie industrielle. '*8)+' |'
    for i in range(80))
long_parts=split_large_table(long_table,'Grand tableau',2)
assert len(long_parts)>1 and all(len(x['text'])<=CHUNK_SIZE for x in long_parts)
assert any('M-079' in p['text'] for p in long_parts)
# Ligne géante : aucun mini-fragment orphelin ne doit apparaître entre deux fragments pleins.
giant_table='| Code | Description |\n| --- | --- |\n| M-001 | '+('Ingénierie industrielle et données scientifiques. '*120)+' |'
giant_parts=split_large_table(giant_table,'Ligne géante',3)
giant_sizes=[len(x['text']) for x in giant_parts]
assert len(giant_parts)>3 and max(giant_sizes)<=CHUNK_SIZE
assert min(giant_sizes[:-1])>CHUNK_SIZE//2, giant_sizes
assert all(x['text'].startswith('Tableau 3, ligne 1 — ') for x in giant_parts)
print('Tests de découpage des tableaux courts et longs : OK',len(long_parts),'fragments',
      '| ligne géante :',len(giant_parts),'fragments de',min(giant_sizes),'à',max(giant_sizes),'caractères')

In [ ]:
from hashlib import sha256
ALL_CHUNKS=[]
for d in INDEX_DOCUMENTS:
    pieces=chunk_structure(d)
    for n,piece in enumerate(pieces,1):
        digest=sha256((d['document_id']+'|'+piece['section']+'|'+piece['text']).encode()).hexdigest()[:16]
        chunk_id=f"{d['document_id']}_{n:04d}_{digest}"
        ALL_CHUNKS.append({
          'chunk_id':chunk_id,'document_id':d['document_id'],
          'domain':d['domain'],'title':d['title'],
          'source_url':d.get('source_url'),'collected_at':d.get('collected_at'),
          'academic_years_detected':d.get('academic_years_detected',[]),
          'dataset_kind':d['dataset_kind'], 'document_role':d.get('document_role','content_page'),
          'section':piece['section'],'chunk_type':piece['kind'],
          'text':piece['text'],'char_count':len(piece['text']),
          'table_index':piece.get('table_index'),
          'table_row_start':piece.get('table_row_start'),
          'table_row_fragment':piece.get('table_row_fragment',False),
          'oversize_table':False,  # conservé pour rétrocompatibilité NB03
        })
assert ALL_CHUNKS
assert len({c['chunk_id'] for c in ALL_CHUNKS})==len(ALL_CHUNKS)
assert set(c['document_id'] for c in ALL_CHUNKS)==set(d['document_id'] for d in INDEX_DOCUMENTS)
assert all(len(c['text'])<=CHUNK_SIZE for c in ALL_CHUNKS)
CHUNKS_DF=pd.DataFrame(ALL_CHUNKS)
display(CHUNKS_DF.groupby(['domain','chunk_type']).agg(chunks=('chunk_id','count'),
                        taille_moyenne=('char_count','mean'),taille_max=('char_count','max')).round(1))
print('Total :',len(ALL_CHUNKS),'chunks ; dépassements :',sum(c['char_count']>CHUNK_SIZE for c in ALL_CHUNKS))
print('Lignes de tableau géantes restructurées :',sum(c['table_row_fragment'] for c in ALL_CHUNKS))

### Visualiser les chunks et leurs métadonnées

Dans le tableau ci-dessous, **`section`** permet d'expliquer *où* l'information a été trouvée. **`source_url`** permettra de produire une citation exacte, si une page officielle a effectivement été collectée.

Le modèle reçoit une version légèrement enrichie du texte (`titre + section + contenu`) ; nous ne l'utilisons pas comme preuve supplémentaire. Seul le **contenu extrait de la source** est considéré comme documentaire.

In [ ]:
display(CHUNKS_DF[['domain','title','section','chunk_type','char_count','oversize_table']].head(15))
ax=CHUNKS_DF['char_count'].plot(kind='hist',bins=min(12,max(2,len(CHUNKS_DF))),figsize=(7,3.1),title='Distribution des tailles des chunks')
ax.set_xlabel('Caractères'); ax.set_ylabel('Nombre de chunks')
plt.tight_layout();plt.show()

def text_for_embedding(chunk):
    return f"Titre : {chunk['title']}\nSection : {chunk['section']}\n{chunk['text']}"

print('Extrait réellement envoyé à l’encodeur :\n')
print(text_for_embedding(ALL_CHUNKS[0])[:650])

### À vous de jouer — expérience de chunking

1. Modifiez `CHUNK_SIZE` à 500 puis 1 000 caractères, en conservant un overlap cohérent.
2. Recalculez `ALL_CHUNKS`. Combien de chunks supplémentaires produisez-vous ?
3. Un tableau d'admission perd-il sa structure ? Montrez le fragment et son titre.
4. Pourquoi un recouvrement inter-sections risquerait-il d'associer les conditions d'admission de deux parcours différents ?

**Ne confondez pas caractère et token :** ce notebook contrôle la taille en caractères. Le nombre de tokens dépend de l'encodeur utilisé.

---
## Étape 3 — Embeddings multilingues

Un encodeur transforme un texte en un vecteur numérique :

\[e_i = f_\theta(\text{titre}_i,\text{section}_i,\text{chunk}_i)\in\mathbb{R}^{d}\]

Nous utilisons **`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`**, adapté à une première expérience en français. La dimension `d` sera lue directement depuis le modèle (et non supposée).

Lorsque les vecteurs sont normalisés, le produit scalaire correspond à la similarité cosinus :

\[\operatorname{cos}(u,v)=\frac{u\cdot v}{\lVert u\rVert_2\,\lVert v\rVert_2}\]

Les embeddings ne contiennent **ni l'URL ni la vérité administrative**. Les URLs et métadonnées sont conservées à côté du vecteur, dans le *payload* Qdrant.

📌 Le premier chargement du modèle nécessite Internet et peut être long sur CPU. Aucune clé API n'est nécessaire.

In [ ]:
EMBEDDING_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
BATCH_SIZE = 16

encoder = SentenceTransformer(EMBEDDING_MODEL)
texts_to_encode=[text_for_embedding(c) for c in ALL_CHUNKS]
VECTORS = encoder.encode(texts_to_encode, batch_size=BATCH_SIZE, show_progress_bar=True,
                         convert_to_numpy=True, normalize_embeddings=True)
VECTORS=np.asarray(VECTORS,dtype=np.float32)
assert VECTORS.ndim==2 and VECTORS.shape[0]==len(ALL_CHUNKS)
DIM=int(VECTORS.shape[1])
assert np.all(np.isfinite(VECTORS))
assert np.allclose(np.linalg.norm(VECTORS,axis=1),1.0,atol=1e-3)
print('Matrice :',VECTORS.shape, '| dimension :',DIM)
print('Premières coordonnées du premier chunk :',VECTORS[0,:8].round(4))

### Diagnostic — ce que l'encodeur lit réellement

Deux limites de cette version de référence doivent être **mesurées**, pas supposées :

1. **Troncature.** La limite du modèle s'exprime en *tokens* (`encoder.max_seq_length`), alors que `CHUNK_SIZE` est en caractères. Tout ce qui dépasse la limite est ignoré par l'encodeur : un chunk de 800 caractères précédé de `Titre + Section` peut n'être représenté que par son début.
2. **Doublons.** Une page « Mention » recopie souvent le texte de ses parcours : le même passage peut alors occuper plusieurs rangs du top-k.

Ce notebook **conserve** ce découpage pour servir de référence ; l'Atelier 02+ (Notebook 02 V3) corrige la troncature et compare les deux index.

In [ ]:
MODEL_MAX_TOKENS=int(encoder.max_seq_length)
sample_idx=np.unique(np.linspace(0,len(ALL_CHUNKS)-1,min(400,len(ALL_CHUNKS)),dtype=int))
token_counts=np.array([len(encoder.tokenizer.encode(texts_to_encode[i],add_special_tokens=True,truncation=False))
                       for i in sample_idx])
n_truncated=int((token_counts>MODEL_MAX_TOKENS).sum())
print(f'Limite du modèle : {MODEL_MAX_TOKENS} tokens | chunks échantillonnés : {len(sample_idx)}')
print(f'Chunks tronqués par l’encodeur : {n_truncated} ({n_truncated/len(sample_idx):.0%}) '
      f'| médiane : {int(np.median(token_counts))} tokens | maximum : {int(token_counts.max())}')

def text_key(text):
    """Empreinte d'un texte normalisé : sert à repérer les passages recopiés d'une page à l'autre."""
    return hashlib.sha256(re.sub(r'\s+',' ',text).strip().casefold().encode()).hexdigest()

key_counts=Counter(text_key(c['text']) for c in ALL_CHUNKS)
n_dup=sum(n-1 for n in key_counts.values() if n>1)
print(f'Chunks au texte strictement identique à un autre : {n_dup} / {len(ALL_CHUNKS)}')
if n_truncated:
    print('→ Ces chunks sont indexés d’après leur début seulement : voir le Notebook 02 V3.')

In [ ]:
# Échantillonnage plafonné : l'ancien graphique pouvait atteindre >100 000 px.
QUESTIONS_EXPLORATION=[
  'Quelles formations abordent les capteurs et la maintenance intelligente ?',
  'Quels sont les documents requis pour candidater en master ?',
  'Quels sont les thèmes scientifiques des laboratoires et projets de recherche ?',
]
Q_VECTORS=encoder.encode(QUESTIONS_EXPLORATION,normalize_embeddings=True,convert_to_numpy=True)
SIM=np.asarray(Q_VECTORS) @ VECTORS.T
indices_base=np.linspace(0,SIM.shape[1]-1,min(80,SIM.shape[1]),dtype=int)
indices_top=np.concatenate([np.argsort(-row)[:10] for row in SIM])
indices=np.array(sorted(set(indices_base.tolist()+indices_top.tolist())))
if len(indices)>120: indices=indices[:120]
fig,ax=plt.subplots(figsize=(12,3.6))
im=ax.imshow(SIM[:,indices],aspect='auto',vmin=-1,vmax=1)
ax.set_yticks(range(len(QUESTIONS_EXPLORATION)),['Formations','Admissions','Recherche'])
ax.set_xlabel('Échantillon de chunks (indices dans la matrice complète)')
ax.set_title(f'Similarités question ↔ chunks : {len(indices)} affichés / {SIM.shape[1]}')
fig.colorbar(im,ax=ax,label='similarité cosinus')
plt.tight_layout();plt.show()
print('Ce sous-ensemble sert à visualiser, il ne constitue pas un échantillon statistique aléatoire.')

---
## Étape 4 — Indexer dans Qdrant

**Une collection** Qdrant regroupe des points ayant le même schéma vectoriel. **Un point** possède un identifiant, son embedding et un **payload** documentaire (`title`, `section`, `domain`, `source_url`, etc.).

Nous utilisons la **distance COSINE** et une base locale persistante. Les données restent dans le stockage temporaire Colab jusqu'à leur export.

⚠️ **Relancer la cellule d'indexation recrée uniquement la collection pédagogique nommée ci-dessous.** N'utilisez pas cette opération de suppression/recréation sur une base de production.

In [ ]:
COLLECTION=f"uppa_{DATASET_KIND}_chunks_v1"
# Dans un runtime Colab, libérer l'ancienne connexion avant de réouvrir la même base locale.
if 'client' in globals():
    try: client.close()
    except Exception: pass
client=QdrantClient(path=str(DB_DIR))
if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)
client.create_collection(collection_name=COLLECTION,
    vectors_config=models.VectorParams(size=DIM,distance=models.Distance.COSINE))

points=[]
for chunk,vector in zip(ALL_CHUNKS,VECTORS):
    # Le mode local accepte un identifiant entier ou un UUID standard.
    point_id=str(uuid.uuid5(uuid.NAMESPACE_URL,chunk['chunk_id']))
    payload=dict(chunk)
    payload['embedding_model']=EMBEDDING_MODEL
    points.append(models.PointStruct(id=point_id,vector=vector.tolist(),payload=payload))
for start in range(0,len(points),64):
    client.upsert(collection_name=COLLECTION,points=points[start:start+64],wait=True)

info=client.get_collection(COLLECTION)
print('Collection :',COLLECTION)
print('Points stockés :',info.points_count)
assert info.points_count==len(ALL_CHUNKS)

### Le payload : ce que l'on peut filtrer et citer

Contrairement à une base de tableaux, **Qdrant stocke ici une représentation vectorielle et des métadonnées**. Nous pourrons filtrer `domain = admission` avant la recherche, puis afficher l'URL officielle et le titre de la section.

Sur de petits corpus de cours, un filtrage local est suffisant ; dans une architecture serveur, on pourrait ajouter des index de payload adaptés aux filtres fréquents.

In [ ]:
sample_point=client.retrieve(collection_name=COLLECTION,ids=[points[0].id],with_payload=True,
                              with_vectors=False)[0]
print('UUID :',sample_point.id)
print('Domaine :',sample_point.payload['domain'])
print('Section :',sample_point.payload['section'])
print('Source :',sample_point.payload.get('source_url') or '(fictive, non officielle)')
print('Texte du chunk :\n',sample_point.payload['text'][:480])
assert sample_point.payload['dataset_kind']==DATASET_KIND

---
## Étape 5 — Rechercher les passages pertinents

La question est convertie avec **le même modèle** que les documents. Qdrant retrouve ensuite les `k` embeddings les plus proches.

Deux cas :

- **Recherche globale** : tous les chunks sont candidats.
- **Recherche filtrée** : seuls les chunks dont le payload respecte `domain = formation`, `admission` ou `recherche` sont candidats.

Les passages au **texte identique** (pages qui se recopient) ne sont renvoyés qu'une fois ; passez `deduplicate=False` pour observer l'effet des doublons.

Le **score cosinus** mesure une proximité vectorielle ; ce n'est **pas** une probabilité de véracité, ni une preuve qu'une condition d'admission est applicable.

In [ ]:
def retrieve(question, domain='tous', k=4, deduplicate=True):
    if not isinstance(question,str) or not question.strip(): raise ValueError('Question vide')
    if domain not in ALLOWED_DOMAINS | {'tous'}: raise ValueError('Domaine inconnu')
    if k<1: raise ValueError('k doit être positif')
    q=np.asarray(encoder.encode(question,normalize_embeddings=True),dtype=np.float32)
    filt=None
    if domain != 'tous':
        filt=models.Filter(must=[models.FieldCondition(key='domain',
                           match=models.MatchValue(value=domain))])
    # Sur-échantillonner puis écarter les textes identiques (pages qui se recopient) :
    # sans cela, le même passage peut occuper plusieurs rangs du top-k.
    limit=min(k*4+8 if deduplicate else k,len(ALL_CHUNKS))
    result=client.query_points(collection_name=COLLECTION,query=q.tolist(),
                               query_filter=filt,limit=limit,with_payload=True)
    hits=[];seen=set()
    for hit in result.points:
        payload=dict(hit.payload or {})
        key=text_key(payload.get('text',''))
        if deduplicate and key in seen: continue
        seen.add(key)
        hits.append({'rank':len(hits)+1,'score':float(hit.score),'point_id':str(hit.id),**payload})
        if len(hits)>=k: break
    return hits


def show_retrieval(question,domain='tous',k=4,preview_chars=280):
    results=retrieve(question,domain,k)
    print(f'QUESTION : {question}\nFILTRAGE : {domain} | Résultats : {len(results)}')
    rows=[]
    for r in results:
        rows.append({'rang':r['rank'],'score_cosinus':round(r['score'],4),
                     'domaine':r['domain'],'titre':r['title'],'section':r['section'],
                     'extrait':r['text'][:preview_chars],
                     'source':r.get('source_url') or '(exemple fictif)'})
    display(pd.DataFrame(rows))
    return results

results_formation = show_retrieval('Quelles formations concernent les systèmes intelligents ?',domain='formation',k=3)
assert all(r['domain']=='formation' for r in results_formation)
assert all(r['dataset_kind']==DATASET_KIND for r in results_formation)

In [ ]:
EXEMPLES_METIER = [
 {'master':'Industry 4.0','question':'Quelles formations abordent la maintenance prédictive, les capteurs ou les systèmes industriels ?', 'domaine':'formation'},
 {'master':'Industry 4.0','question':'Quels laboratoires et axes scientifiques concernent les systèmes cyber-physiques ?', 'domaine':'recherche'},
 {'master':'Logistique et services','question':'Quelles formations concernent les systèmes d’information, la logistique et les services ?', 'domaine':'formation'},
 {'master':'Logistique et services','question':'Quelles recherches portent sur la traçabilité et l’optimisation d’entrepôt ?', 'domaine':'recherche'},
 {'master':'Transversal','question':'Comment candidater à un master et quels documents sont demandés ?', 'domaine':'admission'},
]
for ex in EXEMPLES_METIER:
    print('\n'+'═'*78+'\n'+ex['master'])
    hits=retrieve(ex['question'],domain=ex['domaine'],k=2)
    for r in hits:
        print(f" {r['rank']}. [{r['score']:.3f}] {r['title']} / {r['section']}")

# Les résultats ci-dessus sont des passages retrouvés, PAS des réponses générées.
# Si les sources sont fictives, aucune information n'est attribuable à l'UPPA.

### Question transversale : ne pas imposer un seul domaine

Exemple : *« Quelles formations abordent l'informatique industrielle, comment candidater et quels laboratoires travaillent sur des sujets voisins ? »*

Pour une question transversale, commencez par interroger **tous les domaines**, puis observez si les résultats couvrent formations, admissions et recherche. Un seul `top-k` global peut privilégier un domaine au détriment des autres : cela motive la **recherche multi-domaines** et la fusion de résultats, abordées au Notebook 03/04.

In [ ]:
QUESTION_TRANSVERSALE = ('Quelles formations abordent les systèmes industriels, comment candidater '
                         'et quels laboratoires s’intéressent aux systèmes intelligents ?')
hits_global=show_retrieval(QUESTION_TRANSVERSALE,domain='tous',k=min(6,len(ALL_CHUNKS)))
print('Domaines couverts par le top-k global :',sorted({r['domain'] for r in hits_global}))


def retrieve_diversified(question,k=6,domains=None,max_per_document=2):
    """Retrieval dense diversifié : garantit des candidats de chaque domaine disponible.

    Les scores entre domaines restent ceux de Qdrant, mais la sélection est
    par quotas, PAS un classement global probabiliste. Ne pas confondre avec RRF.
    """
    domains=list(domains or ('formation','admission','recherche'))
    if any(x not in ALLOWED_DOMAINS for x in domains) or len(set(domains))!=len(domains):
        raise ValueError('Domaines invalides ou dupliqués')
    if not domains or k<1: raise ValueError('Recherche vide')
    candidates={d:retrieve(question,domain=d,k=min(max(k*4,8),len(ALL_CHUNKS))) for d in domains}
    selected=[]; seen=set(); counts=Counter()
    # Premier passage : équité entre domaines tant que la couverture est possible.
    while len(selected)<k:
        progress=False
        for domain in domains:
            for hit in candidates[domain]:
                uid=hit['chunk_id']
                if uid in seen or counts[hit['document_id']]>=max_per_document: continue
                selected.append(hit);seen.add(uid);counts[hit['document_id']]+=1
                progress=True
                break
            if len(selected)>=k:break
        if not progress:break
    for rank,hit in enumerate(selected,1):hit['rank']=rank
    return selected

hits_diversifies=retrieve_diversified(QUESTION_TRANSVERSALE,k=6,max_per_document=2)
print('Domaines couverts par la sélection diversifiée :',sorted({h['domain'] for h in hits_diversifies}))
display(pd.DataFrame([{'rang':h['rank'],'domaine':h['domain'],'titre':h['title'],
                       'section':h['section'],'score_cosinus':round(h['score'],4),
                       'source':h.get('source_url') or '(fictif)'} for h in hits_diversifies]))
assert len({h['chunk_id'] for h in hits_diversifies})==len(hits_diversifies)
assert all(sum(h['document_id']==d for h in hits_diversifies)<=2 for d in {h['document_id'] for h in hits_diversifies})
print('Note : la diversité ne garantit pas la pertinence. Elle doit être évaluée sur des références annotées.')

---
## Étape 6 — Contrôles, audit et qualité du retrieval

**Contrôles techniques** : nombre de points, dimensions, unicité des IDs, présence du domaine, conservation des sections et des tableaux.

**Qualité scientifique** : pour calculer Recall@k, Hit@k ou MRR, il faut des **questions annotées**, associées à des `document_id` ou `chunk_id` pertinents. Le score cosinus seul ne valide pas une réponse.

Nous incluons des exemples de questions d'évaluation **uniquement pour le mode fictif** ; pour un corpus réel, les étudiants doivent constituer leurs jugements de pertinence à partir des fiches officielles effectivement collectées.

In [ ]:
# Questions sur les faits réellement présents : annoter les documents AVANT calcul.
# Aucun document officiel n'est présenté comme pertinent sans validation humaine.
if DATASET_KIND=='synthetic_demo':
    labels=[
      ('Quels cours traitent des capteurs ?', 'Parcours Industry 4.0'),
      ('Quels documents pour une candidature en master ?', 'Admission en master'),
      ('Quel document présente un calendrier de candidature ?', 'Calendrier'),
      ('Quels laboratoires travaillent sur les réseaux et la simulation ?', 'Laboratoire Systèmes numériques'),
      ('Quel projet aborde la traçabilité et les entrepôts ?', 'Projet Logistique intelligente'),
    ]
    relevance=[]
    for question,title_part in labels:
        expected_ids=[d['document_id'] for d in INDEX_DOCUMENTS if title_part.lower() in d['title'].lower()]
        if expected_ids:
            relevance.append({'question':question,'reference_document_ids':expected_ids})
else:
    relevance=[]
    real_questions=[
      ('formation','Quels parcours sont proposés dans le Master Informatique ?'),
      ('formation','Quels programmes abordent les systèmes industriels et les capteurs ?'),
      ('formation','Quels objectifs et compétences présente le parcours Industry 4.0 ?'),
      ('formation','Quelles sont les spécificités du parcours SIGLIS ?'),
      ('admission','Quelle procédure de candidature est indiquée pour le master 1 ?'),
      ('admission','Quelle procédure est indiquée pour une candidature en master 2 ?'),
      ('admission','Quelles dates de candidature sont précisées pour une année donnée ?'),
      ('recherche','Quels laboratoires sont liés à l’informatique et aux systèmes intelligents ?'),
      ('recherche','Quelles thématiques scientifiques sont décrites pour les laboratoires ?'),
      ('transversal','Quelles formations abordent les systèmes industriels, comment candidater et quels laboratoires pourraient être pertinents ?'),
    ]
    annotation_template=pd.DataFrame([{
        'question_id':f'Q{i:02d}','domain_cible':domain,'question':question,
        'reference_document_ids':'',  # renseigner un ou plusieurs IDs séparés par ;
        'justification':'', 'statut_annotation':'a_annoter'
    } for i,(domain,question) in enumerate(real_questions,1)])
    annotation_path=EXPORT_DIR/'questions_a_annoter_uppa.csv'
    annotation_template.to_csv(annotation_path,index=False,encoding='utf-8-sig')
    # Catalogue d'IDs pour retrouver sans ambiguïté les sources à annoter.
    source_lookup=pd.DataFrame([{
        'document_id':d['document_id'], 'domain':d['domain'],
        'title':d['title'], 'source_url':d.get('source_url')
    } for d in INDEX_DOCUMENTS])
    source_lookup.to_csv(EXPORT_DIR/'documents_a_annoter_uppa.csv',index=False,encoding='utf-8-sig')
    print('Grille de questions :',annotation_path)
    print('Index des documents pour renseigner les IDs :',EXPORT_DIR/'documents_a_annoter_uppa.csv')
    display(annotation_template[['question_id','domain_cible','question','reference_document_ids']])


def evaluate_document_retrieval(rows,k=3,use_diversification=False):
    allowed={d['document_id'] for d in INDEX_DOCUMENTS}
    observations=[]
    for row in rows:
        if isinstance(row.get('reference_document_ids'),str):
            refs={s.strip() for s in row['reference_document_ids'].split(';') if s.strip()}
        else:
            refs=set(row['reference_document_ids'])
        if not refs:continue
        if not refs.issubset(allowed):
            raise ValueError(f"Références inconnues : {sorted(refs-allowed)}")
        question=row['question']
        hits=(retrieve_diversified(question,k=k) if use_diversification else retrieve(question,domain='tous',k=k))
        # Mesurer la pertinence des DOCUMENTS, pas des chunks répétés.
        unique=[]
        for h in hits:
            if h['document_id'] not in unique: unique.append(h['document_id'])
        ranks=[i for i,docid in enumerate(unique,1) if docid in refs]
        rank=min(ranks) if ranks else None
        observations.append({'question':question,'n_refs':len(refs),'docs_retrouves':len(set(unique)&refs),
                             'Hit@k':int(rank is not None),
                             'Recall@k':len(set(unique)&refs)/len(refs),
                             'MRR@k':1/rank if rank else 0.0})
    result=pd.DataFrame(observations)
    if not result.empty:
        display(result)
        print('Hit@k',round(result['Hit@k'].mean(),3),
              '| Recall@k',round(result['Recall@k'].mean(),3),
              '| MRR@k',round(result['MRR@k'].mean(),3))
    else:
        print('AUCUNE métrique calculée : références humaines non renseignées.')
    return result

if DATASET_KIND=='synthetic_demo':
    report_eval=evaluate_document_retrieval(relevance,k=3)
    print('⚠️ Évaluation sur documents FICTIFS uniquement.')
else:
    report_eval=pd.DataFrame()
    print('Pour évaluer le vrai corpus, compléter questions_a_annoter_uppa.csv puis exécuter :')
    print("labels=pd.read_csv(annotation_path).fillna(''); evaluate_document_retrieval(labels.to_dict('records'),k=3)")

In [ ]:
# Audit reproductible du chunking et de la base.
assert len(ALL_CHUNKS)==len(points)==client.get_collection(COLLECTION).points_count
assert VECTORS.shape[0]==len(ALL_CHUNKS)
assert len(set(p.id for p in points))==len(points)
assert all(c['section'] and c['text'] for c in ALL_CHUNKS)
assert all(c['domain'] in ALLOWED_DOMAINS for c in ALL_CHUNKS)
assert all(c['dataset_kind']==DATASET_KIND for c in ALL_CHUNKS)
assert all(c['source_url'] is not None for c in ALL_CHUNKS) if DATASET_KIND=='live' else all(c['source_url'] is None for c in ALL_CHUNKS)
print('✅ Contrôles réussis : identifiants, dimensions, provenance, nombre de points, chunks.')
print('Matrice embeddings :',VECTORS.shape)
print('Corpus réel validé ?', DATASET_KIND=='live')
assert all(not is_navigation_hub(d) for d in INDEX_DOCUMENTS)
assert all(ch['char_count']<=CHUNK_SIZE for ch in ALL_CHUNKS)
print('✅ Aucun grand tableau conservé comme chunk surdimensionné.')


### Exercices à rendre (les deux masters)

**Socle commun**

1. Comparez un `top-k` global et une recherche filtrée par domaine.
2. Interprétez un résultat élevé en score cosinus mais **non pertinent** pour la question.
3. Justifiez le choix de 800/120 et l'exception de taille pour les tableaux d'admission.
4. Présentez une fiche de chunk : domaine, titre, section, provenance et identifiant.

**Industry 4.0** — Proposez trois questions concernant compétences industrielles, capteurs, automatisation ou axes de recherche associés. Repérez les sections retrouvées et notez les erreurs.

**Systèmes informatiques pour le génie de la logistique industrielle et des services** — Proposez trois questions concernant systèmes d'information, optimisation des services, organisation logistique ou recherche associée. Notez les écarts entre mots exacts et proximités sémantiques.

**Admission** — Choisissez une question nécessitant de distinguer M1/M2, année universitaire ou profil de candidat. Expliquez pourquoi **le score de retrieval ne suffit pas** à valider la procédure. N'inventez aucune date.

📌 Si vous travaillez sur les six fiches fictives, formulez vos questions comme des **tests techniques**, pas comme des faits institutionnels.

---
## Étape 7 — Exporter vers le Notebook 03

Nous enregistrons :

- les **chunks normalisés** (`chunks_*.jsonl`) ;
- un **manifeste** (modèle, taille des chunks, domaine, nombre de points, provenance) ;
- un **rapport de chunking** CSV ;
- des **questions de recherche** CSV, avec leurs scores uniquement lorsqu'elles sont annotées ;
- la **base Qdrant locale persistante**, après fermeture de la connexion.

Tous ces éléments seront placés dans un ZIP. Le Notebook 03 pourra charger le ZIP et rouvrir Qdrant avec `QdrantClient(path=...)` en utilisant **exactement le même modèle d'embeddings**.

**Attention :** les fichiers d'une base locale ne sont pas destinés à des écritures concurrentes. Pour le déploiement réel, préférer un serveur Qdrant configuré et sauvegardé.

In [ ]:
SUFFIX='uppa' if DATASET_KIND=='live' else 'synthetic_demo'
chunks_file=EXPORT_DIR/f'chunks_{SUFFIX}.jsonl'
with chunks_file.open('w',encoding='utf-8') as f:
    for ch in ALL_CHUNKS:
        f.write(json.dumps(ch,ensure_ascii=False)+'\n')

chunking_report=CHUNKS_DF.groupby(['domain','chunk_type'],as_index=False).agg(
    nombre_chunks=('chunk_id','count'),taille_moyenne=('char_count','mean'),taille_max=('char_count','max'))
chunking_report.to_csv(EXPORT_DIR/f'rapport_chunking_{SUFFIX}.csv',index=False,encoding='utf-8-sig')

manifest={
  'schema_version':'uppa_rag_index_v1',
  'created_at':datetime.now(timezone.utc).isoformat(timespec='seconds'),
  'input':INPUT_ORIGIN,'dataset_kind':DATASET_KIND,'source_schema':'uppa_rag_doc_v1',
  'collection_name':COLLECTION,'embedding_model':EMBEDDING_MODEL,
  'embedding_dimension':DIM,'distance':'COSINE','normalized_embeddings':True,
  'chunking':'structure_aware_table_rows_then_recursive','chunk_size_characters':CHUNK_SIZE,
  'chunk_overlap_characters':CHUNK_OVERLAP,
  'n_documents':len(INDEX_DOCUMENTS),'n_input_documents':len(DOCUMENTS),'n_chunks':len(ALL_CHUNKS),
  'domains':dict(Counter(c['domain'] for c in ALL_CHUNKS)),
  'source_urls_verified':DATASET_KIND=='live',
  'is_official_uppa_corpus':DATASET_KIND=='live',
  'note':'Démonstration fictive' if DATASET_KIND!='live' else 'Pages publiques collectées par le Notebook 01 : vérifier leur actualité',
}
manifest_file=EXPORT_DIR/'index_manifest.json'
manifest_file.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
if not report_eval.empty:
    report_eval.to_csv(EXPORT_DIR/f'evaluation_retrieval_{SUFFIX}.csv',index=False,encoding='utf-8-sig')
else:
    pd.DataFrame(columns=['question','n_refs','docs_retrouves','Hit@k','Recall@k','MRR@k']).to_csv(
        EXPORT_DIR/f'evaluation_retrieval_{SUFFIX}.csv',index=False,encoding='utf-8-sig')

assert len([json.loads(line) for line in chunks_file.read_text(encoding='utf-8').splitlines()])==len(ALL_CHUNKS)
print('Exports :',chunks_file.name,manifest_file.name)
print('Mode :',DATASET_KIND)

In [ ]:
# Libérer les fichiers de base avant leur empaquetage.
client.close()
archive_path=BASE/f'UPPA_Notebook02_index_{SUFFIX}.zip'
with zipfile.ZipFile(archive_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(EXPORT_DIR.rglob('*')):
        if path.is_file(): z.write(path,path.relative_to(BASE))
    for path in sorted(DB_DIR.rglob('*')):
        if path.is_file(): z.write(path,path.relative_to(BASE))
print('Archive construite :',archive_path)
print('Taille :',round(archive_path.stat().st_size/1024/1024,2),'Mo')
print('Contenu :',len(zipfile.ZipFile(archive_path).namelist()),'fichiers')
print('Pour télécharger dans Colab, exécuter :')
print(f'  from google.colab import files; files.download({str(archive_path)!r})')

---
## Étape 8 — Préparer le Notebook 03 : recherche + génération avec citations

Le Notebook 03 aura pour entrée **l'archive produite ci-dessus**, pas de nouvelles pages téléchargées. Il rouvrira Qdrant, appliquera des filtres de domaine, préparera les extraits de contexte avec leurs sources, puis transmettra ce contexte à un LLM.

**Avant de rendre le travail :**

- [ ] J'ai indiqué si j'ai utilisé le corpus **officiel** ou **fictif**.
- [ ] J'ai testé deux valeurs de `chunk_size` et expliqué les différences.
- [ ] J'ai montré un tableau correctement conservé.
- [ ] J'ai contrôlé les champs `domain`, `section` et `source_url`.
- [ ] J'ai indexé les embeddings dans Qdrant et montré des recherches filtrées.
- [ ] J'ai créé une archive contenant `index_manifest.json`, les chunks et la base locale.
- [ ] J'ai distingué score de similarité, pertinence documentaire et fiabilité d'une réponse.

### Sources techniques

- [LangChain — RecursiveCharacterTextSplitter](https://reference.langchain.com/python/langchain-text-splitters/character/RecursiveCharacterTextSplitter)
- [Sentence Transformers — Multilingual Models](https://www.sbert.net/examples/sentence_transformer/training/multilingual/README.html)
- [Qdrant Client — Python, local mode et query_points](https://github.com/qdrant/qdrant-client)

> Ce mini-projet est indépendant de l'UPPA et ne constitue pas un service officiel. Pour des réponses sur les admissions, toujours confirmer les procédures en vigueur sur le site institutionnel.